In [1]:
import pandas as pd

df =  pd.read_excel(r"C:\Users\User\Desktop\New Try\HAND GESTURE DETECTION Projest\New folder\New folder\hand_landmarks_updated\SMALL sample for test Hand gesture check.xlsx")

print(df.shape)
df.head()

(322, 213)


,dist_0_1,dist_0_2,dist_0_3,dist_0_4,dist_0_5,dist_0_6,dist_0_7,dist_0_8,dist_0_9,dist_0_10,...,dist_16_20,dist_17_18,dist_17_19,dist_17_20,dist_18_19,dist_18_20,dist_19_20,person_id,person_name,gesture_label
0,0.502450,0.553236,0.601528,0.599021,0.645497,0.679337,0.683784,0.692073,0.688632,0.687614,...,0.378500,0.439188,0.385526,0.247030,0.181678,0.203590,0.277980,175,abhijay,1
1,0.515212,0.559118,0.605037,0.603825,0.646189,0.679813,0.685647,0.694818,0.692034,0.688186,...,0.279923,0.441003,0.421717,0.295331,0.172116,0.135065,0.164600,175,abhijay,1
2,0.532806,0.564553,0.607924,0.608990,0.644356,0.681411,0.686279,0.696041,0.689255,0.689377,...,0.309027,0.467927,0.428264,0.290519,0.194549,0.188953,0.243455,175,abhijay,1
3,0.514383,0.557978,0.601868,0.599224,0.648907,0.681099,0.685859,0.694703,0.693498,0.690021,...,0.347116,0.448481,0.407381,0.273084,0.177207,0.173315,0.227623,175,abhijay,1
4,0.505736,0.550608,0.598556,0.599132,0.643174,0.679151,0.683929,0.693453,0.689384,0.689419,...,0.323146,0.467472,0.408243,0.263451,0.216390,0.241664,0.319505,175,abhijay,1


In [3]:
from sklearn.preprocessing import LabelEncoder

X = df.drop(
    columns=["person_id", "person_name", "gesture_label"]
)

y = df["gesture_label"]

le = LabelEncoder()
y = le.fit_transform(y)

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
from sklearn.svm import SVC

model = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

model.fit(X_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [7]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)

print(f"Accuracy = {acc*100:.2f}%")

Accuracy = 98.46%


In [8]:
import joblib

pipeline = {
    "model": model,
    "scaler": scaler,
    "label_encoder": le
}

joblib.dump(
    pipeline,
    "gesture_pipeline.pkl"
)

print("Saved Successfully")

Saved Successfully


In [9]:
pipeline = joblib.load(
    "gesture_pipeline.pkl"
)

print(pipeline.keys())

dict_keys(['model', 'scaler', 'label_encoder'])


In [10]:
import numpy as np
import pandas as pd
import joblib

In [11]:
def extract_distance_features(landmarks):

    features = []

    for i in range(21):
        for j in range(i + 1, 21):

            x1, y1, z1 = landmarks[i]
            x2, y2, z2 = landmarks[j]

            dist = ((x2-x1)**2 +
                    (y2-y1)**2 +
                    (z2-z1)**2) ** 0.5

            features.append(dist)

    return np.array(features)

In [12]:
pipeline = joblib.load("gesture_pipeline.pkl")

model = pipeline["model"]
scaler = pipeline["scaler"]
label_encoder = pipeline["label_encoder"]

print("Loaded Successfully")

Loaded Successfully


In [13]:
dummy_landmarks = np.random.rand(21,3)

features = extract_distance_features(dummy_landmarks)

print(features.shape)

(210,)


In [14]:
features = features.reshape(1,-1)

features = scaler.transform(features)

pred = model.predict(features)

gesture = label_encoder.inverse_transform(pred)

print("Prediction:", gesture[0])

Prediction: 0


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [28]:
print(type(model))
print(type(scaler))
print(type(label_encoder))
print(label_encoder.classes_)

<class 'sklearn.svm._classes.SVC'>
<class 'sklearn.preprocessing._data.StandardScaler'>
<class 'sklearn.preprocessing._label.LabelEncoder'>
[0 1 2 3 4]


In [34]:
import mediapipe as mp

In [35]:
import mediapipe as mp

print(mp.__version__)
print(mp)

0.10.35
<module 'mediapipe' from 'c:\\Users\\User\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\mediapipe\\__init__.py'>


In [36]:
import mediapipe as mp

print(mp.__file__)

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\mediapipe\__init__.py


In [37]:
import mediapipe as mp

print(mp.__version__)
print(dir(mp))

0.10.35
['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [38]:
import mediapipe as mp

print(mp.__file__)
print(mp.__version__)
print(dir(mp))

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\mediapipe\__init__.py
0.10.35
['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [39]:
dir(mp)

['Image',
 'ImageFormat',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 'tasks']

In [48]:
import mediapipe as mp

print(mp.__version__)
print(dir(mp))
print(hasattr(mp, "solutions"))
print(mp.__file__)

0.10.35
['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']
False
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\mediapipe\__init__.py


In [49]:
import mediapipe as mp

print(mp.__version__)

mp_hands = mp.solutions.hands

print("OK")

0.10.35


AttributeError: module 'mediapipe' has no attribute 'solutions'

In [23]:
import cv2
import mediapipe as mp

ModuleNotFoundError: No module named 'mediapipe'

In [24]:
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Webcam
cap = cv2.VideoCapture(0)

while True:

    success, frame = cap.read()

    if not success:
        break

    # Mirror View
    frame = cv2.flip(frame, 1)

    # BGR -> RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(rgb)

    if results.multi_hand_landmarks:

        for hand_landmarks in results.multi_hand_landmarks:

            # Draw Hand
            mp_draw.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS
            )

            # Count Landmarks
            print(
                "Landmarks:",
                len(hand_landmarks.landmark)
            )

    cv2.imshow("Hand Detection", frame)

    key = cv2.waitKey(1)

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()

NameError: name 'mp' is not defined

In [50]:
import sys

print(sys.executable)

c:\Users\User\AppData\Local\Programs\Python\Python313\python.exe


In [51]:
import mediapipe as mp

print(mp.__version__)
# 0.10.35

print(hasattr(mp, "solutions"))
# False

0.10.35
False


In [53]:
import mediapipe as mp

print(mp.__version__)
print(dir(mp.tasks))

0.10.35
['BaseOptions', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'audio', 'components', 'genai', 'text', 'vision']


In [54]:
mediapipe.tasks.vision

NameError: name 'mediapipe' is not defined